# Uplift modelling, Qini, calibration, and policy value

This notebook loads the cross-fitted out-of-fold scores written by scripts/run_analysis.py (one row per analysis-sample user, with mu0, mu1, and T-learner and X-learner uplift scores, all produced out of fold) and recomputes the evaluation slate: Qini coefficients, decile calibration, and doubly robust policy values at the known constant propensity 0.85. It closes by comparing the recomputed Qini coefficients with the values in reports/results.json.

In [1]:
import sys
sys.path.insert(0, '../src')
import json
import pandas as pd
from uplift_platform.metrics import policy_table, qini_coefficient, uplift_calibration

pd.set_option('display.width', 140)
oof = pd.read_parquet('../data/processed/oof_scores.parquet')
y = oof['conversion'].to_numpy(float)
t = oof['treatment'].to_numpy(int)
print('scored rows:', len(oof))

scored rows: 1000000


## Qini coefficients per estimator

In [2]:
qini = {
    't_learner': qini_coefficient(y, t, oof['tau_t'].to_numpy()),
    'x_learner': qini_coefficient(y, t, oof['tau_x'].to_numpy()),
}
qini

{'t_learner': 0.1331797999688208, 'x_learner': 0.09049618260137889}

## Calibration by predicted decile (X-learner)

In [3]:
pd.DataFrame(uplift_calibration(y, t, oof['tau_x'].to_numpy()))

,decile,n,n_treated,n_control,rate_treated,rate_control,observed_uplift,mean_predicted_uplift,incremental_conversions_in_bin
0,1,100000,86414,13586,0.021964,0.014647,0.007317,1.328390e-02,731.660243
1,2,100000,85207,14793,0.001655,0.001352,0.000303,6.510614e-04,30.280281
2,3,100000,84646,15354,0.000496,0.000326,0.000171,1.578302e-04,17.053607
3,4,100000,86279,13721,0.000243,0.000510,-0.000267,7.230078e-05,-26.677047
4,5,100000,83942,16058,0.000095,0.000125,-0.000029,5.913995e-05,-2.924461
5,6,100000,84732,15268,0.000189,0.000262,-0.000073,5.267529e-05,-7.315519
6,7,100000,85649,14351,0.000315,0.000070,0.000246,2.562780e-05,24.555855
7,8,100000,83737,16263,0.000215,0.000123,0.000092,6.595912e-06,9.198020
8,9,100000,88071,11929,0.000136,0.000168,-0.000031,-3.803460e-07,-3.140494
9,10,100000,82241,17759,0.005496,0.001802,0.003694,-2.897176e-03,369.413886


## Doubly robust policy values

In [4]:
for key in ('tau_t', 'tau_x'):
    table = policy_table(y, t, oof['mu0'].to_numpy(), oof['mu1'].to_numpy(), oof[key].to_numpy(), [0.1, 0.2, 0.3, 0.5], 0.85)
    print(key)
    print(pd.DataFrame(table).T[['value', 'se', 'ci_low', 'ci_high', 'incremental_vs_treat_none']])

tau_t
               value        se    ci_low   ci_high  incremental_vs_treat_none
top_10pct   0.002794  0.000080  0.002637  0.002952                   0.000894
top_20pct   0.002833  0.000077  0.002682  0.002984                   0.000933
top_30pct   0.002833  0.000076  0.002685  0.002981                   0.000932
top_50pct   0.002842  0.000074  0.002697  0.002987                   0.000941
treat_all   0.003058  0.000060  0.002941  0.003174                   0.001157
treat_none  0.001901  0.000107  0.001690  0.002111                   0.000000


tau_x
               value        se    ci_low   ci_high  incremental_vs_treat_none
top_10pct   0.002740  0.000078  0.002586  0.002893                   0.000839
top_20pct   0.002773  0.000074  0.002629  0.002918                   0.000873
top_30pct   0.002791  0.000072  0.002649  0.002933                   0.000890
top_50pct   0.002766  0.000070  0.002629  0.002903                   0.000865
treat_all   0.003058  0.000060  0.002941  0.003174                   0.001157
treat_none  0.001901  0.000107  0.001690  0.002111                   0.000000


## Agreement with reports/results.json

In [5]:
results = json.loads(open('../reports/results.json').read())
for key in ('t_learner', 'x_learner'):
    stored = results['estimators'][key]['qini']['qini_coefficient']
    print(key, 'recomputed:', round(qini[key], 6), 'stored:', round(stored, 6))

t_learner recomputed: 0.13318 stored: 0.13318
x_learner recomputed: 0.090496 stored: 0.090496
